# ENRICH-Aerial — GCP-controlled bundle adjustment


In [ ]:
from pathlib import Path

import numpy as np
import pycolmap
from pycolmap import logging

from pycolmap_ba import backend, frontend, utils

import enrich_data

# --- paths ------------------------------------------------------------------
DATASET = enrich_data.DATA_ROOT / "full"
image_path = DATASET / "images"
database_path = DATASET / "colmap/database.db"
sfm_path = DATASET / "colmap/sparse"
sfm_gcp_path = DATASET / "colmap/sparse_gcp"  # after the controlled bundle

# --- what to run ------------------------------------------------------------
# Feature extraction and brute-force matching take a while on CPU, so by default
# an existing database is reused.
FORCE_OVERWRITE_DB = False
# Constrain the camera centres in the GCP *bundle*, like Metashape's _GT project
# (all 39 references enabled). The camera-centre accuracy row then becomes a fit
# to data the bundle was given - the honest configuration is False.
USE_CAMERA_PRIORS = True
# Let the controlled bundle free the focal length. Safe here, and only here: the
# control is what breaks the flat-block degeneracy.
REFINE_INTRINSICS = True

logging.set_log_destination(logging.INFO, Path.cwd() / "INFO.log")

## 1. Data

`load()` reads the CSVs as they are; `add_noise()` returns another object of the
same type with a simulated survey applied. `truth` is kept for scoring, `data`
is what the pipeline is allowed to see.

In [2]:
truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = truth  # or: enrich_data.add_noise(truth) to simulate a real survey
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, marker_name, u, v
camera_centers = data.camera_xyz
marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

# --- assumed accuracies (see docs: only the ratios matter) ------------------
tie_points_sigma_px = enrich_data.tie_points_sigma_px
gcp_sigma_px = enrich_data.gcp_sigma_px
gcp_sigma_m = enrich_data.gcp_sigma_m
camera_position_sigma_m = enrich_data.camera_position_sigma_m

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 px
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


In [3]:
# Everything the reports need to score the block. Partial truth is fine: each
# field just enables one more row.
gt = utils.GroundTruth(
    camera_xyz=truth.camera_xyz,
    marker_xyz=truth.marker_xyz,
    control_names=truth.control_names,
    check_names=truth.check_names,
    cam_from_world=enrich_data.cam_from_world_all(truth),
    focal_px=enrich_data.FOCAL_PX,
)

<!-- ## 2. Free network

Features, matches, known intrinsics, incremental mapping. Nothing here knows
about the GCPs yet. -->

In [4]:
# print(pycolmap.FeatureExtractionOptions().summary())
# print(pycolmap.FeatureMatchingOptions().summary())

In [5]:
database_path.parent.mkdir(parents=True, exist_ok=True)

reader_options = pycolmap.ImageReaderOptions(
    camera_model="SIMPLE_PINHOLE",
)
extraction_options = pycolmap.FeatureExtractionOptions(
    type=pycolmap.FeatureExtractorType.ALIKED_N16ROT,
    max_image_size=-1,
    aliked=pycolmap.AlikedExtractionOptions(
        max_num_features=4096,
        min_score=0.1,
    ),
    use_gpu=False,
)
pycolmap.extract_features(
    database_path,
    image_path,
    camera_mode=pycolmap.CameraMode.SINGLE,
    reader_options=reader_options,
    extraction_options=extraction_options,
)


I20260814 15:35:52.464397 889836 feature_extraction.cc:494] === Feature extraction ===
I20260814 15:35:52.692611 889854 feature_extraction.cc:270] Processed file [1/339]
I20260814 15:35:52.692646 889854 feature_extraction.cc:273]   Name:            01_backward.jpg
W20260814 15:35:52.692653 889854 feature_extraction.cc:277] 01_backward.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260814 15:35:52.692751 889854 feature_extraction.cc:270] Processed file [2/339]
I20260814 15:35:52.692767 889854 feature_extraction.cc:273]   Name:            01_forward.jpg
W20260814 15:35:52.692777 889854 feature_extraction.cc:277] 01_forward.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260814 15:35:52.692792 889854 feature_extraction.cc:270] Processed file [3/339]
I20260814 15:35:52.692802 889854 feature_extraction.cc:273]   Name:            01_left.jpg
W20260814 15:35:52.692812 889854 feature_extraction.cc:277] 01_left.jpg IMAGE_EXISTS: Features for image were already e

In [9]:
# Add Camera intrinsics to the database
with pycolmap.Database.open(Path(database_path)) as db:
    for entry in db.read_all_cameras():
        entry.model = pycolmap.CameraModelId.SIMPLE_PINHOLE
        entry.width = enrich_data.IMAGE_WIDTH
        entry.height = enrich_data.IMAGE_HEIGHT
        entry.params = np.asarray(
            [enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT], dtype=float
        )
        entry.has_prior_focal_length = True
        db.update_camera(entry)
        print(f"  camera {entry.camera_id}: {entry.model.name} {entry.params}")

  camera 1: SIMPLE_PINHOLE [5865.1810585 3008.        2008.       ]


In [ ]:
# Add camera pose priors to the database

# Add some noise to the camera centres, to simulate a real survey.
rng = np.random.default_rng(42)
noise = rng.normal(0, camera_position_sigma_m, size=(len(camera_centers), 3))
cams_xys = {
    name: xyz + noise[i] for i, (name, xyz) in enumerate(camera_centers.items())
}

sigma = np.broadcast_to(np.asarray(camera_position_sigma_m, dtype=float), (3,))
n_written = 0
with pycolmap.Database.open(Path(database_path)) as db:
    # Priors persist in the database across sessions, and writing a second
    # one for the same image violates a UNIQUE constraint - so reuse the
    # existing id and update in place instead.
    existing = {p.corr_data_id: p.pose_prior_id for p in db.read_all_pose_priors()}
    for image in db.read_all_images():
        if image.name not in cams_xys:
            continue
        prior = pycolmap.PosePrior()
        prior.corr_data_id = image.data_id
        prior.position = np.asarray(cams_xys[image.name], dtype=float)
        prior.position_covariance = np.diag(sigma**2)
        prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
        if image.data_id in existing:
            prior.pose_prior_id = existing[image.data_id]
            db.update_pose_prior(prior)
        else:
            db.write_pose_prior(prior)
        n_written += 1
    n_priors = db.num_pose_priors()

print(f"  wrote {n_written} pose priors, sigma {sigma} m")

  wrote 339 pose priors, sigma [1. 1. 1.] m


In [24]:
# print(pycolmap.FeatureMatchingOptions().summary())
# print(pycolmap.SpatialPairingOptions().summary())
# print(pycolmap.TwoViewGeometryOptions().summary())

In [ ]:
matching_options = pycolmap.FeatureMatchingOptions(
    type=pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE,
    use_gpu=False,
)
pairing_options = pycolmap.SpatialPairingOptions(
    ignore_z=True,
    max_num_neighbors=50,
    min_num_neighbors=0,
    max_distance=100.0,
    num_threads=-1,
)
two_view_geometry_options = pycolmap.TwoViewGeometryOptions(
    min_num_inliers=15,
)
# pycolmap.match_exhaustive(database_path, matching_options=matching_options)


# USe spatial based matching instead of exhaustive
pycolmap.match_spatial(
    database_path,
    matching_options=matching_options,
    pairing_options=pairing_options,
    verification_options=two_view_geometry_options,
)

In [ ]:
import enlighten

reconstruction_manager = pycolmap.ReconstructionManager()
sfm_path.mkdir(parents=True, exist_ok=True)

pipeline_options = pycolmap.IncrementalPipelineOptions()
pipeline_options.use_prior_position = False
pipeline_options.use_robust_loss_on_prior_position = False
pipeline_options.prior_position_loss_scale = float(np.sqrt(7.815))
pipeline_options.ba_refine_focal_length = True
pipeline_options.ba_refine_principal_point = False
pipeline_options.ba_refine_extra_params = False
pipeline_options.mapper.filter_max_reproj_error = 4.0
pipeline_options.mapper.filter_min_tri_angle = 1.5
pipeline_options.mapper.ba_local_min_tri_angle = 6.0

with pycolmap.Database.open(database_path) as db:
    num_images = db.num_images()

with enlighten.Manager() as manager:
    with manager.counter(total=num_images, desc="Images registered:") as pbar:
        pbar.update(0, force=True)
        reconstructions = pycolmap.incremental_mapping(
            database_path,
            image_path,
            sfm_path,
            options=pipeline_options,
            initial_image_pair_callback=lambda: pbar.update(2),
            next_image_callback=lambda: pbar.update(1),
        )

if not reconstructions:
    raise RuntimeError(
        "incremental mapping produced no reconstruction - check the matches "
        "and, if pose priors are enabled, that there are >= 3"
    )
if len(reconstructions) > 1:
    sizes = [r.num_reg_images() for r in reconstructions.values()]
    print(f"  {len(reconstructions)} sub-models, sizes {sizes} -> keeping best")

best_rec = reconstructions[
    max(reconstructions, key=lambda k: reconstructions[k].num_reg_images())
]
print(best_rec.summary())
print("  camera:", next(iter(best_rec.cameras.values())).params)

# best_rec.write(output_path)
best_rec.write_text(sfm_path)

In [ ]:
mappingOptions = frontend.MappingOptions(
    use_prior_position=USE_MAPPER_POSE_PRIORS,
    # A flat nadir block cannot determine focal length from imagery: focal
    # trades against scene depth. Hold the intrinsics at the known values;
    # the GCP bundle can free them later, where the control breaks the
    # degeneracy.
    refine_focal_length=False,
    # COLMAP's own tie-point filter defaults, restated so the tuning surface
    # is visible. All three are NO-OPS on this block - the 80/60 overlap
    # gives strong geometry: measured triangulation angle p1 = 7.4 deg,
    # median 26.8 deg, zero points below 5 deg, max reprojection 3.98 px -
    # but they matter on weaker blocks.
    filter_max_reproj_error=4.0,
    filter_min_tri_angle_deg=1.5,
    ba_local_min_tri_angle_deg=6.0,
)

rec = frontend.run_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=sfm_free_path,
    options=mappingOptions,
    seed=0,
    progress=True,
    write_text=True,
)

In [ ]:
utils.banner("Free network vs the known camera centres")
rmse = utils.camera_center_rmse(rec, camera_centers)
print(f"camera centre RMSE: {rmse:.4f} m")

# A bundle adjustment is a local optimiser: if the network sits in an arbitrary
# frame, the GCP residuals start hopeless and the solve diverges.
# A least-squares similarity over all 39 centres brings it close enough.
if rmse > 1.0:
    utils.align_to_positions(rec, camera_centers)
    print(f"after Sim3: {utils.camera_center_rmse(rec, camera_centers):.4f} m")

    # Overwrite the free network with the aligned version
    utils.export_reconstruction(rec, sfm_free_path)

# A similarity absorbs uniform scale but NOT anisotropic distortion, so a large
# residual that survives the alignment means the free network is deformed -
# almost always a runaway self-calibration - and no control will rescue it.

In [ ]:
# Scored before the control enters, so the effect of the GCPs is measurable.
# No adjusted marker coordinates exist yet, hence the empty dict: only the poses
# and the intrinsics can be scored at this point.
utils.block_report(rec)
utils.accuracy_report(rec, {}, gt, "free network")

## 3. GCP-controlled bundle adjustment

Each marker becomes a parameter block of its own, tied to the poses by its image
marks and - for the control points only - to the survey by a Gaussian prior. The
check points get rays but no prior, so their residual is the one unbiased
accuracy figure in the whole notebook.

All sigmas are given in native units (px, m); the backend expresses them in
tie-point sigma units, because COLMAP adds its tie-point residuals unweighted.

In [ ]:
solverOptions = backend.SolverOptions(
    refine_focal_length=REFINE_INTRINSICS,
    # Points with fewer rays would be IGNORED by the solver, not deleted:
    # they would keep pre-bundle coordinates and still reach the export.
    # 0 = disabled; the post-solve filter below is the one that deletes.
    min_track_length=0,
    # COLMAP's global-BA default. SOFT_L1 @ 1.0 would instead down-weight the
    # reprojection tail (p95 1.77, p99 2.27, max 3.98 px) without discarding
    # redundancy - the soft alternative to hard filtering.
    loss_function_type=None,
    # COLMAP ships function_tolerance = 0, which *disables* the relative
    # stopping test and makes every solve run to the iteration cap and report
    # NO_CONVERGENCE although the cost stopped improving. 1e-8 fixes it.
    function_tolerance=1e-8,
)
markersConstraints = backend.MarkerConstraints(
    observations=marker_data,
    xyz=surveyed_xyz,
    control_names=control_names,  # the check points stay unconstrained
    sigma_px=gcp_sigma_px,
    sigma_m=gcp_sigma_m,
    robust_loss_scale=2.8,  # Cauchy on the marks, to absorb mis-identification
)
camera_posesConstraints = (
    backend.CameraPoseConstraints(
        positions=camera_centers, sigma_m=camera_position_sigma_m
    )
    if USE_CAMERA_PRIORS
    else None
)
filterOptions = backend.FilterOptions(
    max_reproj_error=4.0, min_tri_angle_deg=1.5, min_track_length=0
)

result = backend.run_bundle_adjustment(
    rec,
    solver=solverOptions,
    markers=markersConstraints,
    camera_poses=camera_posesConstraints,
    # Post-solve filtering, same thresholds as the mapper: no-ops here, kept
    # explicit so every threshold in the notebook is visible in one place.
    filters=filterOptions,
    tie_points_sigma_px=tie_points_sigma_px,
    covariance=True,
    verbose=True,
)
marker_xyz = result.marker_xyz  # adjusted marker coordinates

In [ ]:
cov = result.covariance
cov.get_point_cov(1)

In [ ]:
utils.block_report(rec, marker_observations=marker_data, marker_xyz=marker_xyz)
utils.accuracy_report(
    rec, marker_xyz, gt, "after GCP bundle", camera_priors_used=USE_CAMERA_PRIORS
)

## 4. Export

Binary + text model for COLMAP, plus a point cloud carrying track length and
reprojection error per point so the sparse cloud can be filtered in a viewer.

In [ ]:
utils.export_reconstruction(rec, sfm_gcp_path)
utils.save_point_cloud_as_ply(rec, sfm_gcp_path / "sparse_gcp_o3d.ply")